# MNIST con scikit-learn

Arquitectura común: **784 → 128 (ReLU) → 64 (ReLU) → 10**. Adam (`lr=0.001`), entropía cruzada, batch size 64, 10 épocas y semilla 42.

In [ ]:
from pathlib import Path
import json
import sys
import time

import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score, confusion_matrix
from sklearn.neural_network import MLPClassifier

ROOT = Path.cwd().resolve()
if ROOT.name == 'src':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))
from mnist_loader import load_mnist

SEED = 42
BATCH_SIZE = 64
EPOCHS = 10
LEARNING_RATE = 0.001
np.random.seed(SEED)

In [ ]:
X_train, y_train, X_test, y_test = load_mnist(ROOT / 'data')
assert X_train.shape == (60000, 784)
assert y_train.shape == (60000,)
assert X_test.shape == (10000, 784)
assert y_test.shape == (10000,)

fig, axes = plt.subplots(2, 5, figsize=(10, 4))
for image, label, axis in zip(X_train[:10], y_train[:10], axes.flat):
    axis.imshow(image.reshape(28, 28), cmap='gray')
    axis.set_title(f'Etiqueta: {label}')
    axis.axis('off')
plt.tight_layout()
plt.show()

X_train = X_train.astype(np.float32) / 255.0
X_test = X_test.astype(np.float32) / 255.0
y_train = y_train.astype(np.int64)
y_test = y_test.astype(np.int64)

In [ ]:
model = MLPClassifier(
    hidden_layer_sizes=(128, 64),
    activation='relu',
    solver='adam',
    learning_rate_init=LEARNING_RATE,
    batch_size=BATCH_SIZE,
    max_iter=EPOCHS,
    random_state=SEED,
)
start = time.perf_counter()
model.fit(X_train, y_train)
training_time = time.perf_counter() - start
loss_curve = model.loss_curve_

In [ ]:
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
cm = confusion_matrix(y_test, y_pred)
print(f'Accuracy test: {accuracy:.4%}')
print(f'Tiempo de entrenamiento: {training_time:.2f} s')
assert accuracy > 0.90, 'Accuracy inesperadamente baja; revisar el preprocesamiento o entrenamiento.'

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(range(1, len(loss_curve) + 1), loss_curve, marker='o')
axes[0].set(title='Pérdida — scikit-learn', xlabel='Época', ylabel='Cross-entropy')
axes[0].grid(alpha=0.3)
ConfusionMatrixDisplay(cm).plot(ax=axes[1], cmap='Blues', colorbar=False)
axes[1].set_title('Matriz de confusión — scikit-learn')
plt.tight_layout()
plt.show()

results_dir = ROOT / 'results'
results_dir.mkdir(exist_ok=True)
result = {'framework': 'scikit-learn', 'accuracy': accuracy, 'training_time_seconds': training_time, 'loss': list(loss_curve), 'confusion_matrix': cm.tolist()}
with (results_dir / 'sklearn.json').open('w', encoding='utf-8') as file:
    json.dump(result, file, indent=2)